<h1 style="text-align: center;"> Boston Food Safety Conversational AI Data Analyst </h1>

<div style="display: flex; justify-content: space-around;">

<div style="width: 30%; text-align: center;">
<strong>Jie Zhao</strong>  
<br>
jiz273@g.harvard.edu
</div>

<div style="text-align: center; width: 80%; margin: 0 auto;">
    <strong>Abstract</strong><br>
   This project presents a conversational AI analyst for Boston Food Establishment Inspection data using Large Language Models (LLMs), Retrieval-Augmented Generation (RAG), and text-to-SQL techniques. Public inspection datasets contain structured information such as violation codes, severity codes, inspection dates, and business information, as well as unstructured textual comments that are difficult for non-technical users to analyze.
The proposed system allows users to ask natural-language questions about inspection trends, repeated violations, severity patterns, and violation meanings without requiring SQL or data analysis expertise. The architecture combines deterministic SQL analytics for structured numerical queries with semantic retrieval and summarization for contextual inspection comments. The system integrates vector embeddings, semantic search, and tool-calling workflows to provide analytical insights and conversational responses.
This project demonstrates how transformer-based language models and RAG pipelines can improve accessibility and understanding of public health inspection data through an interactive and domain-specific AI analytical system.

</div>

## table of contents

1. [Problem statement](#problem-statement)

2. [Installation, configuration, and setup](#installation-configuration-and-setup)

3. [Dataset and data preparation](#dataset-and-data-preparation)

4. [Project development](#project-development)
   - [4.1 setting up duckdb](#41-setting-up-duckdb)
   - [4.2 sql functions and charts](#42-sql-functions-and-charts)
   - [4.3 embedding and rag setup](#43-embedding-and-rag-setup)
   - [4.4 llm routing and summarization](#44-llm-routing-and-summarization)

5. [Results and demonstration](#results-and-demonstration)
   - [5.1 sql analytics results](#51-sql-analytics-results)
   - [5.2 rag results](#52-rag-results)
   - [5.3 final ask demonstration](#53-final-ask-demonstration)

6. [Benchmarking and evaluation](#benchmarking-and-evaluation)
   - [6.1 text-to-sql vs predefined sql](#62-text-to-sql-vs-predefined-sql)
   - [6.2 plain llm vs rag](#61-plain-llm-vs-rag)
   - [6.3 comparison summary](#63-comparison-summary)

 ## 1. Problem Statement

The Health Division of the Department of Inspectional Services conducts inspections on food establishments in the City of Boston to ensure compliance with sanitary codes and food safety regulations. High-risk establishments often require repeated and follow-up inspections. These inspection activities generate large amounts of public data, including violation descriptions, inspection results, dates, locations, severity levels, and business information.


Despite the availability of this data through public datasets, the information remains difficult for non-technical users to explore and analyze effectively. Extracting meaningful insights typically requires knowledge of SQL, data analytics, or visualization tools. As a result, restaurant owners, city analysts, public health teams, and public member may struggle to answer important operational and public safety questions such as:


- Which establishments repeatedly fail inspections? 

- Which violations are most severe or most common? 

- How has violation issues change over time? 

- What corrective actions should businesses take to improve compliance? 

- Which neighborhoods show higher inspection risk? 



In addition, inspection datasets contain both structured numerical information and unstructured textual comments. Traditional dashboards and static reports are limited in their ability to interpret semantic patterns, summarize recurring issues, or explain contextual relationships between violations and inspection outcomes.


This project addresses these limitations by developing a domain-specific conversational AI analyst capable of processing Boston Food Establishment Inspection data using Retrieval-Augmented Generation (RAG), Large Language Models (LLMs), text-to-SQL analytics, and tool-calling frameworks. The proposed system allows users to ask natural-language questions and receive analytical responses, contextual summaries, data visualizations, and inspection insights without requiring technical expertise.


The system focuses on three primary capabilities:
1.	Contextual Understanding of Inspection Data
The model is enhanced with domain-specific knowledge of violation terminology, severity rankings, inspection standards, and food safety concepts to improve retrieval accuracy and analytical relevance. 


2.	Conversational Analytical Querying
The system combines deterministic SQL analytics with LLM-based reasoning to support flexible natural-language questions regarding trends, severity, repeated violations, and inspection performance. 


3.	Semantic Retrieval and Summarization
Through RAG and vector similarity search, the model analyzes inspection comments and violation descriptions to identify recurring patterns, summarize issues, and provide actionable insights for stakeholders.


## 2. Installation, Configuration and Set UP

### 2.1 Import

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import pyarrow as pa

In [261]:
from dotenv import load_dotenv
# LangChain components for  RAG system
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from sentence_transformers import SentenceTransformer
import duckdb
import subprocess
from pathlib import Path
import faiss




In [96]:
import torch
#enable gpu 

device = "mps" if torch.backends.mps.is_available() else "cpu"
print("Using device:", device)

Using device: mps


In [3]:
# Load your environment variables
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

print("All libraries loaded successfully!")

All libraries loaded successfully!


## 3. Dataset and Data Preparation

Boston Food Establishment Inspections dataset contains the outcomes of food establishment inspections conducted in the city’s greater area since 2006. The dataset is open data source. Updated daily, this dataset provides details about individual inspections and results of businesses serving food. For this analysis, we are working with a static version of the dataset, comprising 27 columns, and 884608 individual records.

The dataset includes information such as the time and location of each inspection, the business entity responsible, and the licensing details. It also records inspection outcomes, violations noted, and any follow-up actions or comments, offering a comprehensive view of Boston’s food safety practices.

### 3.1 Load data and inspect

Boston food establish inspection dataset is downloaded from Boston goverment website 'https://data.boston.gov/dataset/food-establishment-inspections'. The dataset is stored in data folder. Two dataset is used in the project. 
    - food_inspection_report_raw.csv
    - A reference data file for results description. InspectionResult_description.csv

The public dataset documentation did not provide complete descriptions for all columns. Therefore, exploratory data analysis was performed to better understand the dataset structure, column meanings, data types, missing values, uniqueness, and overall data quality before implementing the analytical system.
 

#### 3.1.1. Load dataset using pandas.

In [4]:
#Load and inspect data
df = pd.read_csv('data/food_inspection_report_raw.csv')
print('structure of data:', df.info())


/var/folders/0x/_5wx0swx0vxgsyrkv847t75w0000gn/T/ipykernel_91445/1391034812.py:2: DtypeWarning: Columns (0: zip) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/food_inspection_report_raw.csv')


<class 'pandas.DataFrame'>
RangeIndex: 884608 entries, 0 to 884607
Data columns (total 26 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   businessname  884608 non-null  str    
 1   dbaname       8431 non-null    str    
 2   legalowner    575201 non-null  str    
 3   namelast      884608 non-null  str    
 4   namefirst     508539 non-null  str    
 5   licenseno     884608 non-null  int64  
 6   issdttm       883712 non-null  str    
 7   expdttm       883928 non-null  str    
 8   licstatus     884608 non-null  str    
 9   licensecat    884608 non-null  str    
 10  descript      884608 non-null  str    
 11  result        884608 non-null  str    
 12  resultdttm    878210 non-null  str    
 13  violation     823807 non-null  str    
 14  viol_level    823807 non-null  str    
 15  violdesc      816937 non-null  str    
 16  violdttm      823804 non-null  str    
 17  viol_status   823807 non-null  str    
 18  status_date   3

#### 3.1.2 Inspect data structure, volume, columns.

In [5]:
# review data shape and data stats
print('='*100)
print('size of data', df.shape)
print('='*100)
print('columns', df.columns)
print('='*100)
print('describe of data', df.describe())


size of data (884608, 26)
columns Index(['businessname', 'dbaname', 'legalowner', 'namelast', 'namefirst',
       'licenseno', 'issdttm', 'expdttm', 'licstatus', 'licensecat',
       'descript', 'result', 'resultdttm', 'violation', 'viol_level',
       'violdesc', 'violdttm', 'viol_status', 'status_date', 'comments',
       'address', 'city', 'state', 'zip', 'property_id', 'location'],
      dtype='str')
describe of data            licenseno    property_id
count  884608.000000  727234.000000
mean   107065.636450  150178.259790
std    144712.285047  103322.491624
min        54.000000       0.000000
25%     22153.000000   77703.000000
50%     28531.000000  155991.000000
75%    125523.000000  157956.000000
max    624593.000000  460598.000000


In [380]:
# Look into a few row and inspect the data
pd.set_option("display.max_columns", None)

df.head(5)

,businessname,dbaname,legalowner,namelast,namefirst,licenseno,issdttm,expdttm,licstatus,licensecat,descript,result,resultdttm,violation,viol_level,violdesc,violdttm,viol_status,status_date,comments,address,city,state,zip,property_id,location
0,1000 Degrees Pizza,NaN,KHOSLA VIPAN,Pasquriello LLC,Kenneth Pasquariello,313440,2017-08-14 12:49:37+00,2020-01-01 04:59:00+00,Inactive,FS,Eating & Drinking,HE_Fail,2018-03-20 14:54:25+00,13-2-304/402.11,*,Clean Cloths Hair Restraint,2018-03-20 14:54:25+00,Fail,NaN,One staff person without hair restraint. Provide,55 COURT ST,BOSTON,MA,02108,156226.0,"(42.35925954972639, -71.05890048027378)"
1,1000 Degrees Pizza,NaN,KHOSLA VIPAN,Pasquriello LLC,Kenneth Pasquariello,313440,2017-08-14 12:49:37+00,2020-01-01 04:59:00+00,Inactive,FS,Eating & Drinking,HE_Fail,2018-03-20 14:54:25+00,22-4-601/602.11,**,Food Contact Surfaces Clean,2018-03-20 14:54:25+00,Fail,NaN,Caked on food debris on can opener blade. Clean to remove.,55 COURT ST,BOSTON,MA,02108,156226.0,"(42.35925954972639, -71.05890048027378)"
2,1000 Degrees Pizza,NaN,KHOSLA VIPAN,Pasquriello LLC,Kenneth Pasquariello,313440,2017-08-14 12:49:37+00,2020-01-01 04:59:00+00,Inactive,FS,Eating & Drinking,HE_Fail,2018-03-20 14:54:25+00,M-2-103.11,***,PIC Performing Duties,2018-03-20 14:54:25+00,Fail,NaN,Menu was redesigned allergy statement was removed. Provide proper allery statement for customers to read.,55 COURT ST,BOSTON,MA,02108,156226.0,"(42.35925954972639, -71.05890048027378)"
3,1000 Degrees Pizza,NaN,KHOSLA VIPAN,Pasquriello LLC,Kenneth Pasquariello,313440,2017-08-14 12:49:37+00,2020-01-01 04:59:00+00,Inactive,FS,Eating & Drinking,HE_Filed,2018-08-08 15:54:00+00,08-3-305-307.11,*,Food Protection,2018-08-08 15:54:00+00,Fail,NaN,Several dented cans found on storage shelves. Remove . Set up area for returns PIC removed and is returning to distributor.,55 COURT ST,BOSTON,MA,02108,156226.0,"(42.35925954972639, -71.05890048027378)"
4,1000 Degrees Pizza,NaN,KHOSLA VIPAN,Pasquriello LLC,Kenneth Pasquariello,313440,2017-08-14 12:49:37+00,2020-01-01 04:59:00+00,Inactive,FS,Eating & Drinking,HE_Filed,2018-08-08 15:54:00+00,21-3-304.14,*,Wiping Cloths Clean Sanitize,2018-08-08 15:54:00+00,Fail,NaN,Wet wiping cloths found on counter tops . Remove . Store properly in saniitizer when wet.,55 COURT ST,BOSTON,MA,02108,156226.0,"(42.35925954972639, -71.05890048027378)"


### 3.2  Data columns inspection and transformation
The dataset columns were explored to identify variables relevant to food inspection analysis. This step involved examining each column’s data type, missing values, uniqueness, and cardinality to better understand the dataset structure and data quality.

Inspect violation and description columns: each violation description corresponds to a specific violation code. This helped validate the relationship between the coded inspection fields and the human-readable violation text used later in the analysis.


In [7]:
# inspect violation and description columns
print('violcation code carinality\n\n', df['violation'].value_counts())
print('='*100)
print('violcation description carinality\n\n',df['violdesc'].value_counts())

violcation code carinality

 violation
23-4-602.13            43973
37-6-501.11-.12        39951
15-4-202.16            35183
36-6-501.11-.12        33806
08-3-305-307.11        30211
                       ...  
590.004/4-204.123-C        1
02-3-305.11(2)             1
590.003/3-801.11-C         1
                           1
590.005/5-402.14-PF        1
Name: count, Length: 462, dtype: int64
violcation description carinality

 violdesc
Non-Food Contact Surfaces Clean                                                                   43973
Improper Maintenance of Walls/Ceilings                                                            39951
Non-Food Contact Surfaces                                                                         35183
Improper Maintenance of Floors                                                                    33806
Food Protection                                                                                   30211
                                      

Inspect violation level: column requires to map violation level from stars to understanable severity, and assign numerical scores to severity

In [8]:
# inspect violation level
df['viol_level'].value_counts()

viol_level
*       579434
***     126544
**      110958
-         6869
1919         1
             1
Name: count, dtype: int64

Inspect result column. We can group result into pass, failed, severed failed etc and assign risk scores.

In [9]:
#inspect result column
df['result'].value_counts()

result
HE_Fail       369707
HE_Pass       282095
HE_Filed       92851
HE_FailExt     73291
HE_Hearing     27393
HE_NotReq      23985
HE_TSOP         7669
HE_VolClos      2839
HE_OutBus       2521
Pass             964
HE_Closure       711
Fail             238
HE_FAILNOR       146
HE_Misc          128
DATAERR           42
HE_Hold           17
Failed             6
Closed             2
PassViol           2
NoViol             1
Name: count, dtype: int64

Inspect comment column, we can see the comments usually include the detailed observation of violation, and improvement suggestion.

In [10]:
pd.set_option('display.max_colwidth', None)
df['comments'][:8]

0                                                                               One staff person without hair restraint. Provide
1                                                                     Caked on food debris on can opener blade. Clean to remove.
2                      Menu was redesigned allergy statement was removed. Provide proper allery statement for customers to read.
3    Several dented cans found on storage shelves. Remove . Set up area for returns PIC removed and is returning to distributor.
4                                      Wet wiping cloths found on counter tops . Remove . Store properly in saniitizer when wet.
5                                                                                         No labels on bulk containers . Provide
6                                      Scoops found submerges in flour and other bulk container. Store properly with handles up.
7                                                                                                

In [21]:
#description of inspection result
ref = pd.read_csv('data/InspectionResult_description.csv')
# Standardize column names
ref.columns = ref.columns.str.lower().str.strip()
ref

,inspectionresult,inferreddescription
0,HE_Fail,Inspection failed; violations found
1,HE_Pass,Inspection passed; no issues found
2,HE_Filed,Minor violations found; no urgent follow-up required
3,HE_FailExt,Extended failure from prior inspection
4,HE_Hearing,Violations being addressed; follow-up required
5,HE_NotReq,Inspection not required; no immediate need
6,HE_TSOP,Temporary suspension of permit issued
7,HE_OutBus,Business closed; out of operation
8,HE_VolClos,Voluntary closure by business to avoid penalties
9,HE_Closure,Forced closure due to critical violations


Upon initial inspection of the data, we decided the columns below are relative and meaningful for our anlaysis.

- businessname: business name
- licenseno: Key identifier for the business/ restaurant
-result: result of  inspection
-resultdttm: date on which the results were generated
-violation: coding of law regulation related to violations
- viol_level: level of violation
-violdesc: reason of violation
- violdttm: date on which violation status was generated
- viol_status: status for violation: Fail or Pass
-status_date: date on which violation status was set to pass
- comments: comments given to the food establishment for improvement
- address: address of the business
- zipcode: zipcode of the business
- location: latitude and longitude of the business location

In [12]:
#extract column name, create describtion and decide action to keep or remove for the purpose of the analysis 
columns_name = df.columns
column_description = [
    "business name shown on inspection record.",
    "a different name of the business.",
    "owner of the business.",
    "owner's last name.",
    "owner's first name.",
    "business license number.",
    "license issue date/time.",
    "license expiration date/time.",
    "license status.",
    "license category code.",
    "business category.",
    "inspection result outcome.",
    "inspection result date/time.",
    "violation code.",
    "violation severity level/code.",
    "violation description.",
    "violation date/time.",
    "violation status.",
    "date violation status was updated.",
    "inspector comments or notes.",
    "business street address.",
    "business city.",
    "business state.",
    "business zip code.",
    "property identifier.",
    "geographic location coordinates."
]
column_action = [
    "keep", "remove", "remove", "remove", "remove",
    "keep", "remove", "remove", "remove", "remove",
    "remove", "keep", "keep", "keep", "keep",
    "keep", "keep", "keep", "remove", "keep",
    "keep", "keep", "remove", "keep", "remove", "keep"
]

column_review_df = pd.DataFrame({
    "column_name": columns_name,
    "column_description": column_description,
    "column_action": column_action
})

column_review_df.sort_values(by='column_action')

,column_name,column_description,column_action
0,businessname,business name shown on inspection record.,keep
5,licenseno,business license number.,keep
11,result,inspection result outcome.,keep
12,resultdttm,inspection result date/time.,keep
13,violation,violation code.,keep
14,viol_level,violation severity level/code.,keep
15,violdesc,violation description.,keep
16,violdttm,violation date/time.,keep
17,viol_status,violation status.,keep
19,comments,inspector comments or notes.,keep


In [18]:
cols = column_review_df[column_review_df['column_action'] == 'keep']['column_name']
data =  df[cols]

Tranformation

In [14]:
# mapping violation level from stars to understanable severity, and assign numerical scores to severity
severity_map = {
    "*": "low",
    "**": "medium",
    "***": "high",
    "-": None,
    "1919": None
}
severity_score_map= {
    "*": 1,
    "**": 2,
    "***": 3
}


We aggregate inspection results into category that reflect the meaning of the description and assign risk scores

In [15]:
result_group_map = {
    "HE_Pass": "pass",
    "Pass": "pass",
    
    "HE_Filed": "minor_violation",

    "HE_Fail": "fail",
    "Fail": "fail",
    "Failed": "fail",
    "HE_FAILNOR": "fail",

    "HE_FailExt": "extended_fail",
    "HE_Hearing": "hearing",

    "HE_TSOP": "temporary_suspension",
    "HE_VolClos": "voluntary_closure_avoid",
    "HE_Closure": "forced_closure",

    "HE_OutBus": "out_of_business",
    "Closed": "closed",

    "HE_NotReq": "not_required",
    "HE_Misc": "misc",
    "DATAERR": "data_error",
    "HE_Hold": "hold"
}

risk_score_map = {
    "pass": 0,
    "minor_violation": 2,
    "hold": 2,

    "fail": 3,
    "extended_fail": 3,
    "hearing": 3,

    "temporary_suspension": 4,
    "voluntary_closure_avoid": 4,
    "forced_closure": 4,

    "out_of_business": None,
    "closed": None,
    "not_required": None,
    "misc": None,
    "data_error": None,
}



Map severity code to severity descritpion and map to severity score defined ealier. 
Map result code to risk descritpion and map to risck score defined ealier. 

In [19]:
data["severity_level"] = data["viol_level"].map(severity_map)
data["severity_score"] = data["viol_level"].map(severity_score_map)

data["result_group"] = data["result"].map(result_group_map)
data["risk_score"] = data["result_group"].map(risk_score_map)

In [22]:
# merge the description of inspection result to the main dataframe
data = data.merge(
    ref[["inspectionresult", "inferreddescription"]],
    left_on="result",
    right_on="inspectionresult",
    how="left"
)
data = data.drop(columns=["inspectionresult"])

In [23]:
# the analysis focuses on violation related content, we drop the rows with missing violation, result, viol_level, viol_status
data.dropna(subset=['violation','result','viol_level','viol_status'], inplace=True)


823,807 usable rows,1-2% percent of the data is missing, the field will be filled with unknkown or no comment as description.

In [25]:
# handling na
# need date time data for analysis 
data["resultdttm"] = pd.to_datetime(df["resultdttm"], errors="coerce")
data["resultdttm"] = pd.to_datetime(data["resultdttm"], errors="coerce")
data["year"] = data["resultdttm"].dt.year
data["month"] = data["resultdttm"].dt.strftime("%Y-%m")
data = data.dropna(subset=["resultdttm"])

#remove most current month 2026-05 as it's not completed month
data = data[data['month'] != '2026-05']
# fill na with unknown
data["violdesc"] = data["violdesc"].fillna("Unknown")
data["comments"] = data["comments"].fillna("No comment")
data["address"] = data["address"].fillna("Unknown")
data["inferreddescription"] = data["inferreddescription"].fillna("Unknown")
data["address"] = data["address"].fillna("Unknown")
data["zip"] = data["zip"].astype("string")


In [27]:
data.info()

<class 'pandas.DataFrame'>
Index: 819596 entries, 0 to 884607
Data columns (total 21 columns):
 #   Column               Non-Null Count   Dtype              
---  ------               --------------   -----              
 0   businessname         819596 non-null  str                
 1   licenseno            819596 non-null  int64              
 2   result               819596 non-null  str                
 3   resultdttm           819596 non-null  datetime64[us, UTC]
 4   violation            819596 non-null  str                
 5   viol_level           819596 non-null  str                
 6   violdesc             819596 non-null  str                
 7   violdttm             819594 non-null  str                
 8   viol_status          819596 non-null  str                
 9   comments             819596 non-null  str                
 10  address              819596 non-null  str                
 11  city                 819596 non-null  str                
 12  zip               

In [ ]:
data.head(3)

,businessname,dbaname,legalowner,namelast,namefirst,licenseno,issdttm,expdttm,licstatus,licensecat,...,violdttm,viol_status,status_date,comments,address,city,state,zip,property_id,location
0,1000 Degrees Pizza,NaN,KHOSLA VIPAN,Pasquriello LLC,Kenneth Pasquariello,313440,2017-08-14 12:49:37+00,2020-01-01 04:59:00+00,Inactive,FS,...,2018-03-20 14:54:25+00,Fail,NaN,One staff person without hair restraint. Provide,55 COURT ST,BOSTON,MA,02108,156226.0,"(42.35925954972639, -71.05890048027378)"
1,1000 Degrees Pizza,NaN,KHOSLA VIPAN,Pasquriello LLC,Kenneth Pasquariello,313440,2017-08-14 12:49:37+00,2020-01-01 04:59:00+00,Inactive,FS,...,2018-03-20 14:54:25+00,Fail,NaN,Caked on food debris on can opener blade. Clean to remove.,55 COURT ST,BOSTON,MA,02108,156226.0,"(42.35925954972639, -71.05890048027378)"
2,1000 Degrees Pizza,NaN,KHOSLA VIPAN,Pasquriello LLC,Kenneth Pasquariello,313440,2017-08-14 12:49:37+00,2020-01-01 04:59:00+00,Inactive,FS,...,2018-03-20 14:54:25+00,Fail,NaN,Menu was redesigned allergy statement was removed. Provide proper allery statement for customers to read.,55 COURT ST,BOSTON,MA,02108,156226.0,"(42.35925954972639, -71.05890048027378)"


Write to parquet file for database use.

In [28]:
#write it to parquet file, more efficient for database use
data.to_parquet("data/food_inspections_clean.parquet", engine="pyarrow",index=False)

## 4. Project development

### 4.1 Setting up duckdb

Load cleaned data into DuckDB

In [29]:
# connect to duckdb
con = duckdb.connect("data/food_inspections_clean.duckdb")
# create table
con.execute("""
CREATE OR REPLACE TABLE inspections AS
SELECT *
FROM read_parquet('data/food_inspections_clean.parquet')
""")

con.execute("SELECT COUNT(*) FROM inspections").fetchall()

[(819596,)]

### 4.2 Define SQL functions and Charts

Aggregated data would usually surface interesting insights, we would like have understanding high level around questions below. 
Potential business question: what violation types are most common this year?”

In [254]:
# SQL functions 1 - top violation types,What violations are most common?
def top_violation_types(year=2025, severity_level='high', limit=10):
    # input: year, severity level, limit
    # output: dataframe of violation types and their counts
    # query: select violation type, count of violation
    query = """
    SELECT 
        violdesc,
        COUNT(*) AS violation_count
    FROM inspections
    WHERE violdesc IS NOT NULL 
      AND violdesc <> 'Unknown violation'
      AND result_group <> 'pass'
      AND severity_score IS NOT NULL
      AND (? IS NULL OR EXTRACT(year FROM resultdttm) = ?)
      AND (? IS NULL OR severity_level = ?)
    GROUP BY violdesc
    ORDER BY violation_count DESC
    LIMIT ?
    """
    return con.execute(
        query,
        [year, year, severity_level, severity_level, limit]
    ).df()

Business question: Are high-severity violations increasing faster than mild violations? Distinguishes whether risk is worsening or whether only minor violations are increasing.

In [256]:
# SQL function 2 - violation record trend by year and severity
def violation_count_by_year_and_severity(start_year=2016):
    query = """
    SELECT
        year,
        severity_level,
        COUNT(*) AS violation_record_count
    FROM inspections
    WHERE year >= ?
      AND result_group <> 'pass'
      AND severity_level IS NOT NULL
      AND violdesc IS NOT NULL
      AND violdesc <> 'Unknown violation'
    GROUP BY year, severity_level
    ORDER BY year, severity_level
    """
    return con.execute(query, [start_year]).df()

Business question: Which restaurants repeatedly fail inspections or have consistently severe violations?

For this function, we aim to understand owner violation seriousness. It can computer by severe risk score, serverity level or better, violation count, or better using combined results to generate an overall score. We define a serious violation score to measure how serious a violation pattern is overall for each business. the score combines both violation severity and inspection risk. The score is calculated as: bad_violation_score = AVG(severity_score) * 0.6 + AVG(risk_score) * 0.3+ LOG(COUNT(*) + 1) * 0.1 we give more weight to severity score because it directly reflects how serious the violation is, while risk score provides additional context about the inspection risk level.
We did also include count as factor too as we saw some owners have 32 violation ranks lower then the restrauant with only 4 violation. We use log transformations on counts as the count have bigger range,  for skewed count variables they reduce the influence of very large values while preserving the signal from frequency.

In [257]:
#SQL functions 3 - top violation owner
def top_violation_owner(year=2025, limit=15, rank_by="score"):
    # input: year, limit, rank_by
    # output: dataframe of business name, address, total violations, high severity violation, high risk violation, average severity score, average risk score, average seriousness score
    # query: select business name, address, count of violation, high severity violation, high risk violation, average severity score, average risk score, average seriousness score 
    # serious level of violation is computed using both severity score and risk score
    query = """
    SELECT
        businessname,
        address,
        COUNT(*) AS total_violations,
        SUM(CASE WHEN severity_score >= 3 THEN 1 ELSE 0 END) AS high_severity_count, 
        SUM(CASE WHEN risk_score >= 3 THEN 1 ELSE 0 END) AS high_risk_count, 
        ROUND(AVG(severity_score), 2) AS avg_severity_score,
        ROUND(AVG(risk_score), 2) AS avg_risk_score,
        ROUND(AVG(severity_score) * 0.6 + AVG(risk_score) * 0.3+ LOG(COUNT(*) + 1) * 0.1,2) as overall_violation_score
    FROM inspections
    WHERE businessname IS NOT NULL
      AND severity_score IS NOT NULL
      AND risk_score IS NOT NULL
      AND (? IS NULL OR EXTRACT(year FROM resultdttm) = ?)
    GROUP BY businessname, address
    HAVING COUNT(*) >= 3
    ORDER BY
        CASE
            WHEN ? = 'score' THEN overall_violation_score
            WHEN ? = 'high_count' THEN high_severity_count
            WHEN ? = 'risk_count' THEN high_risk_count
            WHEN ? = 'total_count' THEN total_violations
            ELSE overall_violation_score
        END DESC
    LIMIT ?
    """

    return con.execute(
        query,
        [year, year, rank_by, rank_by, rank_by, rank_by, limit]
    ).df()

Test SQL results

In [258]:
# sql 1 test
top_violation_types(2025)

,violdesc,violation_count
0,(A)(2) and (B) Time/Temperature Control for Safety Food Hot and Cold Holding (P),692
1,Packaged and Unpackaged Food-Separation Packaging and Segregation (P),370
2,(A)(1) Time/Temperature Control for Safety Food Hot and Cold Holding (P),285
3,Manual and Mechanical Warewashing Equipment Chemical Sanitization-Temperature pH Concentration and Hardness (P),188
4,Backflow Prevention (P),170
5,Backflow Prevention Air Gap (P),151
6,System Maintained in Good Repair (P),137
7,Discarding or Reconditioning Unsafe Adulterated or Contaminated Food (P),100
8,When to Wash (P),99
9,Cooling (P),93


In [341]:
top_violation_owner(2024,10)

,businessname,address,total_violations,high_severity_count,high_risk_count,avg_severity_score,avg_risk_score,overall_violation_score
0,Ogawa Coffee,10 MILK ST,5,5.0,3.0,3.00,2.20,2.54
1,El Pupi Chimi,1 CITYWIDE ST,4,1.0,4.0,2.00,4.00,2.47
2,Freeport Street Cafeteria,179 FREEPORT ST,3,3.0,2.0,3.00,2.00,2.46
3,Ethiopian Cafe,377 CENTRE ST,32,11.0,32.0,2.22,3.00,2.38
4,Beantown Kebab,44 KILBY ST,3,2.0,3.0,2.33,3.00,2.36
5,Boston Halal,273 HUNTINGTON AV,3,2.0,3.0,2.33,3.00,2.36
6,Bread Thyme,1868 CENTRE ST,5,3.0,4.0,2.60,2.40,2.36
7,Pho Zabb,1799 COMMONWEALTH AV,6,6.0,3.0,3.00,1.50,2.33
8,El Primo Market,220 COLUMBIA RD,4,4.0,2.0,3.00,1.50,2.32
9,DUNKIN DONUTS(SATELLITE),100 TERMINAL RD,14,4.0,13.0,2.29,2.79,2.32


Define Chart Function

In [260]:
def chart_violation_count_by_year_and_severity(df):
    fig = px.line(
        df,
        x="year",
        y="violation_record_count",
        color="severity_level",
        markers=True,
        title="Violation Record Count by Year and Severity"
    )

    fig.update_layout(
        xaxis_title="Year",
        yaxis_title="Violation Record Count",
        legend_title_text="Severity Level"
    )

    return fig

In [87]:
df_trend = violation_count_by_year_and_severity(2016)
fig = chart_violation_count_by_year_and_severity(df_trend)
fig.show()

In [ ]:
def chart_top_violation_types(df):
    fig = px.bar(
        df,
        x="violation_count",
        y="violdesc",
        orientation="h",
        title="Top Violation Types",
        text="violation_count"
    )

    fig.update_layout(
        yaxis={"categoryorder": "total ascending"},
        xaxis_title="Violation Count",
        yaxis_title="Violation Type"
    )

    return fig

Test chart function

In [204]:
df_top_types = top_violation_types(2025)
chart_top_violation_types(df_top_types)

In [222]:
#chart to show total violation records for each business
def chart_top_violation_owner(df, year=None):
    df = df.copy()

    title_year = f" in {year}" if year else " Overall"

    fig = px.bar(
        df.sort_values("total_violations"),
        x="total_violations",
        y="businessname",
        orientation="h",
        title=f"Top Businesses by Total Violation Records{title_year}",
        hover_data={
            "high_severity_count": True,
            "high_risk_count": True,
            "avg_severity_score": ":.2f",
            "avg_risk_score": ":.2f",
            "overall_violation_score": ":.2f",
            "businessname": False
        }
    )

    fig.update_layout(
        template="plotly_white",
        height=500,
        title_x=0.5,
        xaxis_title="Total Violation Records",
        yaxis_title="Business"
    )

    return fig

In [208]:
df_owner = top_violation_owner(year=2024, limit=15)
chart_top_violation_owner(df_owner, year=2024)

In [212]:
def chart_owner_risk_scatter(df, year=None):
    df = df.copy()

    title_year = f" in {year}" if year else " Overall"

    fig = px.scatter(
        df,
        x="total_violations",
        y="overall_violation_score",
        size="high_risk_count",
        color="avg_severity_score",
        hover_name="businessname",
        hover_data={
            "high_severity_count": True,
            "high_risk_count": True,
            "avg_risk_score": ":.2f",
            "avg_severity_score": ":.2f",
            "overall_violation_score": ":.2f"
        },
        title=f"Business Violation Volume vs Overall Violation Score{title_year}"
    )

    fig.update_layout(
        template="plotly_white",
        height=600,
        title_x=0.5,
        xaxis_title="Total Violation Records",
        yaxis_title="Overall Violation Score"
    )

    return fig

In [218]:
df_owner = top_violation_owner(year=2025, limit=20, rank_by="score")
fig = chart_owner_risk_scatter(df_owner, year=2025)
fig.show()

### 4.3. Embeddings and Rag setup

DataFrame row
→ formatted text document
→ embedding vector
→ FAISS index
→ retrieve similar rows
→ LLM summary

Prepare document



From the closer look at the sample: the same violation comments can occur in both Pass and Fail records, so severity depends on broader inspection context, not only the text. It indicates the need for using both structured fields plus comments together.

In [345]:
data[["comments", "risk_score", "severity_score", "violation",'result','viol_status']].head(20)

,comments,risk_score,severity_score,violation,result,viol_status
0,One staff person without hair restraint. Provide,3.0,1.0,13-2-304/402.11,HE_Fail,Fail
1,Caked on food debris on can opener blade. Clean to remove.,3.0,2.0,22-4-601/602.11,HE_Fail,Fail
2,Menu was redesigned allergy statement was removed. Provide proper allery statement for customers to read.,3.0,3.0,M-2-103.11,HE_Fail,Fail
3,Several dented cans found on storage shelves. Remove . Set up area for returns PIC removed and is returning to distributor.,2.0,1.0,08-3-305-307.11,HE_Filed,Fail
4,Wet wiping cloths found on counter tops . Remove . Store properly in saniitizer when wet.,2.0,1.0,21-3-304.14,HE_Filed,Fail
5,No labels on bulk containers . Provide,2.0,1.0,02-3-602.11-.12/3-302.12,HE_Filed,Fail
6,Scoops found submerges in flour and other bulk container. Store properly with handles up.,2.0,1.0,10-3-304.12,HE_Filed,Fail
10,One staff person without hair restraint. Provide,0.0,1.0,13-2-304/402.11,HE_Pass,Pass
11,Caked on food debris on can opener blade. Clean to remove.,0.0,2.0,22-4-601/602.11,HE_Pass,Pass
12,Menu was redesigned allergy statement was removed. Provide proper allery statement for customers to read.,0.0,3.0,M-2-103.11,HE_Pass,Pass


Prepare documentation for RAG by combining meanningful fields into one text block for each records.

In [92]:
#prepare document for each row, only extract meaningful fields 
def build_rag_text(row):
    fields = {
        "business": row.get("businessname"),
        "address": row.get("address"),
        "zip": row.get("zip"),
        "violation": row.get("violdesc"),
        "severity": row.get("severity_level"),
        "inspection result": row.get("result_group"),
        "inspection result code": row.get("result"),
        "inspection result meaning": row.get("inferreddescription"),
        "violation_status": row.get("viol_status"),
        "Inspector Observation / Corrective Action": row.get("comments")
    }
    # join the fields into a single text
    parts = []
    for label, value in fields.items():
        if pd.notna(value) and str(value).strip() != "":
            parts.append(f"{label}: {value}")

    return "\n".join(parts)

In [93]:
# Create rag dataframe from cleaned data
rag_df = data.copy()
# build rag text
rag_df["rag_text"] = rag_df.apply(build_rag_text, axis=1)

 Generate embedding
 
The embedding model converted inspection text into dense vector representations for semantic similarity search.
consideration: choose the embedding model that balance cost and efficiency and performance
Experiment with lightweight model all-MiniLM-L6-v2, fast, resource-efficient embeddings.  With just 22M parameters, it delivers solid performance on general semantic search tasks and is used across many production-grade apps.



In [ ]:
#combine all text into a list
texts = rag_df["rag_text"].tolist()
#initiate embedding model
embed_model = SentenceTransformer(
    "all-MiniLM-L6-v2",
    device=device
)
# apply embedding to the text
embeddings = embed_model.encode(
    texts,
    batch_size=128
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/6404 [00:00<?, ?it/s]

 model loader saw an extra key but did not need to use it. For sentence-transformers/all-MiniLM-L6-v2, theload report is usually harmless.

In [ ]:
#save embeddings and rag_df to parquet file for future use
np.save("data/rag_embeddings.npy", embeddings)
rag_df.to_parquet("data/rag_df.parquet")

Build FAISS index

FAISS was used as the vector index to enable efficient similarity retrieval over embedded inspection records.

In [104]:

embeddings = embeddings.astype("float32")
print("embeddings shape", embeddings.shape)

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)
print("FAISS index size:", index.ntotal)
# write index to file
faiss.write_index(index, "data/faiss_index.index")

embeddings shape (819596, 384)
FAISS index size: 819596


 RAG retrieval

In [368]:
def retrieve_similar_violations(query, k=10):
    #convert query to same embedding
    query_embedding = embed_model.encode(
        #embeds query to a list as faiss expect 2D array
        [query],
        normalize_embeddings=True
    ).astype("float32")
    #search for top k similar violations and store scores and indices
    scores, indices = index.search(query_embedding, k)
    #get the results
    results = rag_df.iloc[indices[0]].copy()
    results["similarity_score"] = scores[0]

    return results[
        [
            "similarity_score",
            "businessname",
            "address",
            "violdesc",
            "severity_level",
            "result_group",
            "result",
            "inferreddescription",
            "comments"
        ]
    ]

The step test whether the retrieval system returns relevant inspection records for a sample user question. 

In [114]:
query = "Find similar violation patterns related to food spoilation"

retrieved_df = retrieve_similar_violations(query,5)
retrieved_df

,similarity_score,businessname,address,violdesc,severity_level,result_group,result,inferreddescription,comments
350337,0.608371,Halal Indian Cuisine,736 HUNTINGTON AV,(A) Equipment Food-Contact Surfaces Nonfood-Contact Surfaces and Utensils (Pf),medium,fail,HE_Fail,Inspection failed; violations found,Interior of house model refrigerator unitv observed soiled with dry food spills upright freezer unit observed with heavy ice build up cook range observed heavily soiled with dry food encrusted heavy carbon build up and food spills interior of oven observed soiled interior of middle compartment of 3 bay sink observed soiled by dishes and foods front of the house soda refrigerator shelvings observed soiled wall behind cookline observed heavily soiled by dry food spills multiple kitchen shelvings observed soiled exterior of microwave all condiment holders observed soiled . Clean and sanitize all the above equipment and surfaces.
108875,0.604384,BRIGHAM CIRCLE CHINESE FOOD,728 HUNTINGTON AV,(A) Equipment Food-Contact Surfaces Nonfood-Contact Surfaces and Utensils (Pf),medium,fail,HE_Fail,Inspection failed; violations found,Interior and exterior of multiple refrigeration units (shelving/ gastkets/floor) observed soiled exterior of numerous storage containers observed heavily soiled with dry foods encrust and visible food spills cook range observed heavily soiled chest refrigerator unit observed with heavy ice build up and food spills interior and exterior of microwave exterior of multiple rice cooker exterior of squeeze bottles and condiment containers observed soiled (clean and relabel ) and in between equipments observed with dry food encrusted. Clean and sanitize to remove.
509853,0.601675,Montecristo Mexican Grille,748A HUNTINGTON AV,(A) Equipment Food-Contact Surfaces Nonfood-Contact Surfaces and Utensils (Pf),medium,fail,HE_Fail,Inspection failed; violations found,Interior of 4 chest freezer observed without heavy ice build up interior of 2 door refrigerator unit observed soiled with visible food spills exterior of bulk bins observed soiled by visible food spills and crumbs interior of beverage unit observed soiled shelvings inside walkin unit observed soiled and rusty interior of middle compartment of 3 bay sink observed soiled from foods exterior of multiple foods contact surfaces observed soiled by visible food spills.. Clean and sanitize to remove
509903,0.595236,Montecristo Mexican Grille,748A HUNTINGTON AV,(A) Equipment Food-Contact Surfaces Nonfood-Contact Surfaces and Utensils (Pf),medium,extended_fail,HE_FailExt,Extended failure from prior inspection,Interior of 4 chest freezer observed without heavy ice build up interior of 2 door refrigerator unit observed soiled with visible food spills exterior of bulk bins observed soiled by visible food spills and crumbs interior of beverage unit observed soiled shelvings inside walkin unit observed soiled and rusty interior of middle compartment of 3 bay sink observed soiled from foods exterior of multiple foods contact surfaces observed soiled by visible food spills.. Clean and sanitize to remove
109145,0.587940,BRIGHAM CIRCLE CHINESE FOOD,728 HUNTINGTON AV,(A) Equipment Food-Contact Surfaces Nonfood-Contact Surfaces and Utensils (Pf),medium,pass,HE_Pass,Inspection passed; no issues found,Interior and exterior of multiple refrigeration units (shelving/ gastkets/floor) observed soiled exterior of numerous storage containers observed heavily soiled with dry foods encrust and visible food spills cook range observed heavily soiled chest refrigerator unit observed with heavy ice build up and food spills interior and exterior of microwave exterior of multiple rice cooker exterior of squeeze bottles and condiment containers observed soiled (clean and relabel ) and in between equipments observed with dry food encrusted. Clean and sanitize to remove.


RAG Context Formatting
Retrieved records were formatted into structured context before being passed to the LLM.

In [115]:
def format_rag_context(results_df, max_rows=5):
    rows = []

    for i, row in results_df.head(max_rows).iterrows():
        rows.append(f"""
            Record {i+1}
            Business: {row.get("businessname")}
            Violation: {row.get("violdesc")}
            Severity: {row.get("severity_level")}
            Result: {row.get("result_group")}
            Inspection Result: {row.get("inferreddescription")}
            Comment: {row.get("comments")}
            """.strip())

    return "\n\n".join(rows)

In [116]:
format_rag_context(retrieved_df)

'Record 350338\n            Business: Halal Indian Cuisine\n            Violation: (A) Equipment  Food-Contact Surfaces  Nonfood-Contact Surfaces  and Utensils (Pf)\n            Severity: medium\n            Result: fail\n            Inspection Result: Inspection failed; violations found\n            Comment: Interior of house model refrigerator unitv observed soiled with dry food spills  upright freezer unit observed with heavy ice build up  cook range observed heavily soiled with dry food encrusted heavy carbon build up  and food spills  interior of oven observed soiled  interior of middle compartment of 3 bay sink observed soiled by dishes and foods  front of the house soda refrigerator shelvings observed soiled  wall behind cookline observed heavily soiled by dry food spills  multiple kitchen shelvings observed soiled  exterior of microwave  all condiment holders observed soiled . Clean and sanitize all the above equipment and surfaces.\n\nRecord 108876\n            Business: BRIGH

RAG Prompt and LLM Chain.

Use langhchain template to build summary prompt. good prompt will use two or more of them: 
* Instructions
* External information or context
* User input or query
* Output indicator

In [119]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.2 #lower temperature means the model is less random and more consistent.most deterministic, better for routing SQL tool selection.
)

Build RAG prompt

In [356]:
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """
            You are a food safety inspection analyst.
            Use only the retrieved inspection records.
            Do not add causes, recommendations, or facts unless supported.
            If the records do not provide enough evidence, say so.
            Keep the answer concise and evidence-based.
            """),
    ("human", """
            User question:
            {question}

            Retrieved inspection records:
            {context}

            Answer using this format:

            1. Common pattern:
            2. Evidence from retrieved records:
            3. Practical corrective actions supported by the records:

""")
])

rag_chain = rag_prompt | llm | StrOutputParser()

Build Rag summary function. 

It takes the records returned by the RAG retrieval function, formats them as context, and passes them to the LLM for grounded summarization.

In [357]:
def summarize_rag(question, retrieved_df):
    context = format_rag_context(retrieved_df)

    summary = rag_chain.invoke({
        "question": question,
        "context": context
    })

    return summary

In [358]:
query = "Find violations similar to sanitizer or dish machine problems,and provide practical corrective actions"

retrieved_df = retrieve_similar_violations(query, k=5)

response = summarize_rag(query, retrieved_df)

print(response)

1. Common pattern: Issues with sanitizer use and dish machine functionality.

2. Evidence from retrieved records: 
   - Vintage Lounge: Staff not using sanitizer in the low-temperature dishwasher, with critical violations including improper chemical labeling and unsanitary conditions.
   - CAFE MIRROR: Low-temperature dish machine not testing for sanitizer, with instructions to sanitize utensils in a 3-compartment sink and to discontinue using the dish machine until it can sanitize safely at 50ppm.
   - Right Taste Jamaican Restaurant: Wiping cloths not stored in a sanitizing solution, with recommendations for separate buckets for raw and ready-to-eat food contact surfaces.

3. Practical corrective actions supported by the records:
   - Ensure that the low-temperature dish machine is properly serviced and tested for sanitizer concentration, maintaining a level of 50ppm.
   - Use the 3-compartment sink for sanitizing utensils and pans until the dish machine is functional.
   - Set up se

In [364]:
# test the fummary function and the rag retrieval function
query = "What recurring corrective actions appear in Boston inspection comments related to low-temperature dish machines?"

retrieved_df = retrieve_similar_violations(query, 5)
context = format_rag_context(retrieved_df)

response = rag_chain.invoke({
    "question": query,
    "context": context
})

print(response)

1. Common pattern: Recurring issues with high-temperature dish machines not reaching required sanitization temperatures and needing repairs.

2. Evidence from retrieved records: 
   - Record 776047 and Record 776322 both indicate that the high-temperature dish machine was in use but required a new heating element and was taken out of service until repairs were made.
   - Record 94367 and Record 94351 show that the high-temperature warewashing machine did not reach the necessary temperature of 149F, leading to a call for repairs and the use of an alternative machine.

3. Practical corrective actions supported by the records: 
   - Taking the malfunctioning machines out of service until properly repaired, as noted in multiple records.
   - Utilizing alternative warewashing machines while repairs are pending, as indicated in Records 776047, 776322, and 94351.


In [148]:
retrieved_df[
    [
        "similarity_score",
        "businessname",
        "violdesc",
        "severity_level",
        "comments"
    ]
].head(5)

,similarity_score,businessname,violdesc,severity_level,comments
776046,0.706270,The Bostonian Boston - A Millennium Hotel,Food Contact Surfaces Clean,high,High temperature main dish machine - 140F wash 140F rinse new heating element on order. Machine still in use during inspection / Machine will now be taken out of service until properly repaired / 3 bay sink will be used in the meantime Bar low temp glass machine - Sanitizer not registering / Machine taken out of service until properly repaired 3 bay sink will be used.
776321,0.703605,The Bostonian Boston - A Millennium Hotel,Food Contact Surfaces Clean,high,High temperature main dish machine - 140F wash 140F rinse new heating element on order. Machine still in use during inspection / Machine will now be taken out of service until properly repaired / 3 bay sink will be used in the meantime Bar low temp glass machine - Sanitizer not registering / Machine taken out of service until properly repaired 3 bay sink will be used.
101864,0.700029,Boston Park Plaza (Main Kitchen/Room Serv.),(A) and (C) Good Repair and Calibration-Utensils and Temperature and Pressure Measuring Devices (C),low,A service call was conducted on 01/19/23 for repairs to be made to the high temperature machine on the main level. Following that service it was determined that multiple parts needed to be replaced and parts are on order. Email confirmation of service completed on 01/19/23 Facility has large high temperature dish machne in the kitchen on the second level and all warewashing is being done in that machine until proper repairs are made to the machine in the main kitchen.
94366,0.676362,Boston Chops Downtown,Mechanical Warewashing Equipment Hot Water Sanitization Temperatures (Pf),medium,At the high temperature warewashing machine in the barrel room the temperature at the plate using the location's irreversible thermometer did not go above 149F after four attempts. The person in charge placed a call to get the unit repaired and the staff in the barrel room will utilize a different location's warewashing machine. All other high temperature dish machines were observed working properly.
94350,0.675533,Boston Chops Downtown,Mechanical Warewashing Equipment Hot Water Sanitization Temperatures (Pf),medium,At the high temperature warewashing machine in the barrel room the temperature at the plate using the location's irreversible thermometer did not go above 149F after four attempts. The person in charge placed a call to get the unit repaired and the staff in the barrel room will utilize a different location's warewashing machine. All other high temperature dish machines were observed working properly.


Build SQL prompt

In [330]:
sql_summary_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are an experienced food safety data analyst.

Use only the provided SQL result.
Do not invent numbers,names, years, causes, or trends.
If the SQL result is limited, say so.
Keep the answer concise.
Use 2–4 short bullet points maximum.

For business interpretation:
- Explain what the result means for food safety operations.
- Focus on priority, risk, compliance, monitoring, or follow-up action.
- Do not claim causes unless the SQL result supports them.
"""),
    ("human", """
        User question:
        {question}

        SQL tool used:
        {tool_name}

        SQL result:
        {result}

Write the answer using this format:

1. Direct answer:
2. Key evidence from SQL result:
3. Business interpretation:
4. Limitation:
""")
])

sql_summary_chain = sql_summary_prompt | llm | StrOutputParser()

Build SQL summary function

It takes the records returned by the SQL query formats them as context, and passes them to the LLM for grounded summarization.

In [325]:
#create a wrapper function that runs a SQL tool, converts the dataframe to text
def sql_summary(question, tool_name, result,max_rows=80):
    result_text = result.head(max_rows).to_string(index=False) #prevent output huge amount of data rows

    summary = sql_summary_chain.invoke({
    "question": question,
    "tool_name": tool_name,
    "result": result_text
})
    return summary


Test sql summary function

In [326]:
tool_name = "top_violation_types"
df_result = top_violation_types()
question_test1 = "Show me most common violation types in 2024"

summary = sql_summary(question_test1, tool_name, df_result)
print(summary)

1. Direct answer: The most common violation type in 2024 is "Time/Temperature Control for Safety Food Hot and Cold Holding (P)" with 692 counts.  
2. Key evidence from SQL result: The top violation types include:  
   - Time/Temperature Control for Safety Food Hot and Cold Holding (P): 692  
   - Packaged and Unpackaged Food-Separation Packaging and Segregation (P): 370  
   - (A)(1) Time/Temperature Control for Safety Food Hot and Cold Holding (P): 285  
3. Business interpretation: There is a significant focus on time and temperature control violations, indicating potential areas for improvement in food safety practices.  
4. Limitation: The SQL result only provides data for 2024 and does not include historical trends or comparisons to previous years.


adding instruction: Evidence from retrieved records and Practical corrective actions supported by the records produce more ground base anwsers compared to previous promt usage.


In [327]:
question_sql1 = "Show violation record count trends by year and severity since 2016."

df_result = violation_count_by_year_and_severity(start_year=2016)

summary = sql_summary(
    question=question_sql1,
    tool_name="violation_count_by_year_and_severity",
    result=df_result
)

print(summary)

1. Direct answer: Violation record counts show a general decline in high severity violations and fluctuations in low and medium severity violations from 2016 to 2026.

2. Key evidence from SQL result:
   - High severity violations decreased from 6758 in 2016 to 1327 in 2026.
   - Low severity violations fluctuated but remained high, with 28597 in 2016 and 7125 in 2026.
   - Medium severity violations increased from 2181 in 2016 to 3750 in 2026.

3. Business interpretation: The significant reduction in high severity violations suggests improved compliance or effectiveness of food safety measures, while the variability in low and medium severity violations indicates ongoing challenges in those areas.

4. Limitation: The SQL result does not provide a complete trend analysis for all years beyond 2026, limiting insights into future patterns.


Build Router template
Router template: when user ask a question, the router determine which tool to use sql or rag or chart for anwsering questions.
User question - router_chain chooses tool

  if SQL tool:
    run SQL function
    show chart
    sql_summary_chain explains result

if similar_violations:
    run RAG retrieval
    rag_summary_chain explains retrieved records

In [219]:
# LLM router template to determine which tool to use
router_template = """
You are a routing assistant for a food safety analytics system.

Choose exactly one tool:

top_violation_types
- Use for questions about the most common violation types or top violation descriptions.

violation_count_by_year_and_severity
- Use for questions about violation record trends over time, year-based trends, or trends split by severity.

top_violation_owner
- Use for questions about restaurants/businesses with repeated violations, high overall violation scores, high-risk counts, high-severity counts, serious outcomes, repeat offenders, or businesses with many violations.

similar_violations
- Use for semantic search questions asking for similar violations, related issues, corrective actions, examples, inspection comments, or violation patterns.


unknown
- Use if the question does not match any tool.

User question:
{text}

Return only the tool name.
"""

router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["text"]
)

router_chain = router_prompt | llm | StrOutputParser()

In [289]:
# build function to choose which type of ranking to use
def infer_rank_by(question):
    q = question.lower()

    if "overall" in q or "combined" in q or "score" in q:
        return "score"
    if "high severity" in q or "severe" in q:
        return "high_count"
    if "high risk" in q or "risk count" in q:
        return "risk_count"
    if "most violations" in q or "many violations" in q or "repeat" in q:
        return "total_count"

    return "score"

In [265]:
#build route question function
def route_question(question):
    route = router_chain.invoke({"text": question})
    return route

The final ask() function integrates the complete analytical workflow. It receives a natural-language user question, routes the question to the appropriate tool, executes either the selected SQL analytics function or the RAG retrieval function, optionally generates a visualization, and returns the final output.

The returned output includes the selected route, tool type, dataframe result, optional chart, and LLM-generated summary.
question: natural-language user question

year: optional year filter for SQL queries

limit: number of SQL rows returned

k: number of retrieved RAG records

show_chart: whether to generate a visualization

chart_type: chart selection option

In [370]:
#build function to execute the route and return the anwser
def ask(question, year=None, limit=10, k=10, show_chart=True, chart_type="auto"):
    #route to route_question tool to determine which tool to use, output tool name
    route = route_question(question).strip()

    #if tool is top_violation_types, run top_violation_types sql tool
    if route == "top_violation_types":
        df = top_violation_types(year=year, limit=limit)
        #if show_chart is true, show chart
        chart = None
        if show_chart:
            chart = chart_top_violation_types(df)
        #use sql_summary function to summarize the result
        summary = sql_summary(question, route, df)

        #return tool name, type of the tool (sql or rag), data details, summary of the result
        return {
            "route": route,
            "type": "sql",
            "data": df,
            "chart": chart,
            "summary": summary
        }
    #if tool is top_violation_owner, run top_violation_owner sql tool
    elif route == "top_violation_owner":
        #decide which type of ranking to use
        rank_by = infer_rank_by(question)
        df = top_violation_owner(
        year=year,
        limit=limit,
        rank_by=rank_by
    )
        #Two type of charts to choose, scatter plot to show risk score and count plot to show total count only
        chart = None
        if chart_type == "count":
            chart = chart_top_violation_owner(df, year=year)
        elif chart_type == "risk_scatter":
            chart = chart_owner_risk_scatter(df, year=year)
        elif chart_type == "auto":
            if "risk" in question.lower() or "score" in question.lower() or "severity" in question.lower():
                chart = chart_owner_risk_scatter(df, year=year)
            else:
                chart = chart_top_violation_owner(df, year=year)
        summary = sql_summary(question, route, df)
        #return tool name, type of the tool (sql or rag), data details, summary of the result
        return {
            "route": route,
            "type": "sql",
            "data": df,
            "chart": chart,
            "summary": summary
        }
    elif route == "violation_count_by_year_and_severity":
        start_year = year if year is not None else 2016
        df = violation_count_by_year_and_severity(start_year=start_year)

        chart = None
        if show_chart:
            chart = chart_violation_count_by_year_and_severity(df)

        summary = sql_summary(question, route, df)

        return {
            "route": route,
            "type": "sql",
            "data": df,
            "chart": chart,
            "summary": summary
        }

    #if tool is similar_violations, run similar_violations rag tool
    elif route == "similar_violations":
        df = retrieve_similar_violations(question, k=k)
        #use summarize_rag function to summarize the result
        summary = summarize_rag(question, df)
        return {
            "route": route,
            "type": "rag",
            "data": df,
            "summary": summary
        }
    #if tool is not found, return error
    else:
        return {
            "route": route,
            "type": "error",
            "data": None,
            "summary": "No matching tool found."
        }

## 5. Results and Deomonstration

In [ ]:
user_question_1 =  "What are the top 10 businesses with the most violations in 2025?"
user_question_2 =  "Show me most common violation types in 2024"
user_question_3 =  "How have food inspection violations volume changed across severity levels since 2016?"
user_question_4 =  "Which businesses had the highest overall serious violation risk in 2025 that requires follow up?"

user_question_8 = "Find violations similar to sanitizer or dish machine problems,and provide practical corrective actions"
user_question_9 = "What recurring corrective actions appear in Boston inspection comments related to low-temperature dish machines and provide corrective action?"

In [268]:
#test1
result_1 = ask(
    user_question_1,
    year=2025,
)

print(result_1["summary"])
result_1["chart"].show()

1. Direct answer: The top 10 businesses with the most violations in 2025 are led by Dans Mini Dogs, followed by Boston Restaurant Bar & Grill and New York Pizza 433.

2. Key evidence from SQL result:
   - Dans Mini Dogs: 371 total violations
   - Boston Restaurant Bar & Grill: 126 total violations
   - New York Pizza 433: 105 total violations

3. Business interpretation: Dans Mini Dogs has a significantly higher number of violations compared to the other businesses, indicating potential systemic issues in food safety practices.

4. Limitation: The SQL result only provides data for 2025 and does not include historical trends or comparisons to previous years.


The system was able to determine the type of question, select the appropriate analytical tool, generate the SQL output, and produce a chart showing the top 10 businesses with the most violations in 2025. The response included specific numerical references from the dataset and did not invent unsupported facts or statistics.

In [320]:
#test2
result_2 = ask(
    user_question_2,
    year=2024,
)

print(result_2["summary"])
result_2["chart"].show()

1. Direct answer: The most common violation type in 2024 is "Time/Temperature Control for Safety Food Hot and Cold Holding (P)" with 673 occurrences.

2. Key evidence from SQL result: 
   - "Time/Temperature Control for Safety Food Hot and Cold Holding (P)" - 673
   - "Packaged and Unpackaged Food-Separation Packaging and Segregation (P)" - 375
   - Other notable violations include Backflow Prevention Device and System Maintained in Good Repair.

3. Business interpretation: The high number of violations related to time/temperature control indicates a critical area for food safety training and compliance efforts, suggesting a need for improved monitoring and staff education.

4. Limitation: The SQL result only provides data for 2024 and does not include historical trends or comparisons to previous years.


The system selected the SQL violation type analysis tool and identified the most common violation types in 2024. The response summarized the highest-frequency violation categories using actual count values from the SQL result, while the chart provided a quick visual comparison of the top violation types.

In [331]:
#test3
result_3 = ask(
    user_question_3,
)

print(result_3["summary"])
result_3["chart"].show()

1. Direct answer: The trend of violation volume by severity levels shows a decline in high severity violations and fluctuating medium and low severity violations since 2016.

2. Key evidence from SQL result:
   - High severity violations decreased from 6758 in 2016 to 1327 in 2026.
   - Low severity violations have varied but generally decreased from 28597 in 2016 to 7125 in 2026.
   - Medium severity violations increased from 2181 in 2016 to 3750 in 2026.

3. Business interpretation:
   - The significant decline in high severity violations indicates improved compliance and risk management in food safety operations.
   - Continuous monitoring of medium severity violations is necessary as they have shown an upward trend, which may require further investigation and follow-up actions.

4. Limitation:
   - The SQL result does not provide a comprehensive view of the factors influencing these trends or the context behind the changes in violation counts.


Inspect sql resulst and validate the summary is correct.

In [ ]:

df_result[df_result["severity_level"] == "high"]

,year,severity_level,violation_record_count
0,2016.0,high,6758
3,2017.0,high,4991
6,2018.0,high,6937
9,2019.0,high,4022
12,2020.0,high,1820
15,2021.0,high,2951
18,2022.0,high,3213
21,2023.0,high,2520
24,2024.0,high,3233
27,2025.0,high,3088


The system selected the trend analysis tool and grouped violation records by year and severity level starting from 2016. The line chart helped show how violation counts changed over time across severity categories, while the summary explained the main trend without restating every yearly value. The dataframe sql returned is also inspected and the records match the output.

In [336]:
#test4
result_4 = ask(
    user_question_4,
    limit=20, 
    year=2025,
)

print(result_4["summary"])
result_4["chart"].show()

1. Direct answer: The businesses with the highest overall serious violation risk in 2025 that require follow-up are Bakey, Ruth's Chris Steak House, and Taqueria Los Compadres Mexican Food.

2. Key evidence from SQL result:
   - Bakey: overall violation score of 2.69, 12 total violations, 10 high severity.
   - Ruth's Chris Steak House: overall violation score of 2.22, 26 total violations, 11 high severity.
   - Taqueria Los Compadres Mexican Food: overall violation score of 2.23, 37 total violations, 9 high severity.

3. Business interpretation:
   - These businesses exhibit a significant number of serious violations, indicating a high risk for food safety issues.
   - Immediate follow-up actions are necessary to address compliance and mitigate potential health risks.

4. Limitation:
   - The SQL result is limited to the year 2025 and does not provide historical data for trend analysis or context.


In [342]:
#test4
result_6 = ask(
    user_question_6

)

print(result_6["summary"])


1. Common pattern: Recurring issues with low-temperature dish machines not registering sanitizer and high-temperature dish machines requiring repairs.

2. Evidence from retrieved records: 
   - Record 776047 and Record 776322 both indicate that the bar low-temperature glass machine had sanitizer not registering and was taken out of service until properly repaired. 
   - Record 101865 mentions that multiple parts needed to be replaced for the high-temperature machine, with repairs pending.

3. Practical corrective actions supported by the records: The low-temperature glass machine should be repaired to ensure sanitizer is registering properly, and the high-temperature dish machine should be repaired and calibrated following the replacement of necessary parts.


The system selected the business-risk ranking tool and used the engineered overall violation score to compare businesses in 2025. This result demonstrated how severity, inspection risk, and repeated violation frequency could be combined to identify businesses that may need attentions.

In [371]:
#test4
result_8 = ask(
    user_question_8

)

print(result_8["summary"])

1. Common pattern: Issues with sanitizer use and dishwashing equipment.

2. Evidence from retrieved records: 
   - Vintage Lounge records indicate a lack of sanitizer in the low-temperature dishwasher and improper testing of sanitizer levels.
   - CAFE MIRROR records show the low-temperature dish machine not testing for sanitizer and the need to sanitize utensils and pans in a 3-compartment sink until the machine is serviced.
   - Right Taste Jamaican Restaurant record highlights the need for proper storage of wiping cloths in a sanitizing solution.

3. Practical corrective actions supported by the records:
   - For Vintage Lounge and CAFE MIRROR, discontinue using the low-temperature dish machine for sanitizing until it can safely sanitize at 50ppm. Use the 3-compartment sink for sanitizing utensils and pans in the interim.
   - For Right Taste Jamaican Restaurant, set up separate buckets of sanitizer for wiping cloths used on raw and ready-to-eat food contact surfaces, label the buck

In [373]:
#test4
result_9 = ask(
    user_question_9

)

print(result_9["summary"])

1. Common pattern: Recurring issues with high-temperature dish machines and low-temperature glass machines not functioning properly, leading to service calls and repairs.

2. Evidence from retrieved records: 
   - Record 776047 and Record 776322 both indicate that the high-temperature main dish machine had a new heating element on order and was taken out of service until properly repaired. The bar low-temperature glass machine was also noted for sanitizer not registering and was taken out of service.
   - Record 101865 and Record 101802 mention that repairs were needed for the high-temperature machine in the main kitchen, with parts on order and warewashing being done in a different machine until repairs were completed.

3. Practical corrective actions supported by the records: 
   - Ensure timely repairs of high-temperature and low-temperature dish machines, as indicated by the need for parts and service calls.
   - Utilize alternative washing methods (e.g., 3-bay sink) while machines

The system used semantic retrieval to find inspection records similar to sanitizer or dish machine problems. This demonstrated that RAG can retrieve related violations even when the wording in comments differs across records.

The system used RAG to analyze broader food spoilage, food handling, and contamination-related comment patterns. The response showed how free-text inspection comments can be converted into useful pattern insights.

## 6. Benchmark and Evaluation

6.1 LLM-generated text-to-SQL vs Predefined SQL function_llm
This evaluation compared an LLM-generated text-to-SQL workflow with the predefined SQL functions developed for the project. The purpose was to test whether the LLM could translate natural-language questions into executable DuckDB SQL queries and summarize the query results in plain language.


Inspect retrieved data result from sql

6.1 Build text to sql prompt. The prompt includes table schema, columns description, business rules.

In [338]:
text_to_sql_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a DuckDB SQL expert.

Table: inspections

Schema:
- businessname: restaurant/business name
- address: business address
- zip: ZIP code
- resultdttm: inspection/result datetime
- year: extracted inspection year
- month: extracted inspection month
- violdesc: violation description
- severity_level: human-readable severity label
- severity_score: numeric severity score; higher = more severe
- result: raw inspection result code
- result_group: normalized result category
- risk_score: numeric inspection risk score; higher = higher risk
- comments: inspector comments / corrective action notes

Rules:
- Generate SELECT SQL only.
- Return only raw SQL text.
- Filter out unknown or missing violation descriptions.
- overal seriousness considering serverity, violation counts, risk level applying ROUND(AVG(severity_score) * 0.6 + AVG(risk_score) * 0.3+ LOG(COUNT(*) + 1) * 0.1,2) as overall_violation_score
- Do not include result_group = 'pass' for violation trend questions.
- Do not include ```sql or ```.
"""),
    ("human", """
User question:
{question}
""")
])

text_to_sql_chain = text_to_sql_prompt | llm | StrOutputParser()

6. Build text to chart generation

In [227]:
chart_code_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a Python Plotly chart creator.

Given a pandas dataframe schema and user question, generate Python code to create a Plotly chart.

Rules:
- Use plotly.express as px.
- Assume dataframe is named df.
- Return only Python code.
- Do not include markdown.
- Do not reinvent data that doesn't exist.
- Do not modify df.
"""),
    ("human", """
User question:
{question}

Dataframe columns:
{columns}
""")
])

chart_code_chain = chart_code_prompt | llm | StrOutputParser()

Generate result

In [250]:
def run_text_to_sql_benchmark(question):
    # ask the LLM to generate SQL from the user question
    generated_sql = text_to_sql_chain.invoke({
        "question": question
    }).strip()

    # run the generated SQL against DuckDB
    result_df = con.execute(generated_sql).df()
    # Summarize the SQL result using the SQL summary chain
    summary = sql_summary(
        question=question,
        tool_name="text_to_sql_generated_query",
        result=result_df
    )

    # Return all benchmark outputs for comparison
    return {
        "question": question,
        "generated_sql": generated_sql,
        "data": result_df,
        "summary": summary
    }

text to sql LLM test 1:

In [270]:
text_sql_result1 = run_text_to_sql_benchmark(user_question_1)

print(text_sql_result1["generated_sql"])
print(text_sql_result1["summary"])


SELECT businessname, COUNT(violdesc) AS violation_count
FROM inspections
WHERE year = 2025 AND violdesc IS NOT NULL
GROUP BY businessname
ORDER BY violation_count DESC
LIMIT 10;
1. Direct answer: The top 10 businesses with the most violations in 2025 are listed below.
2. Key evidence from SQL result:
   - Dans Mini Dogs: 376 violations
   - Dunkin Donuts: 249 violations
   - Caffe Nero: 225 violations
   - Subway: 208 violations
   - Chilacates: 164 violations
   - Mcdonalds: 130 violations
   - New York Pizza: 130 violations
   - Boston Restaurant Bar & Grill: 126 violations
   - Dunkin' Donuts: 117 violations
   - Tatte Bakery & Cafe: 116 violations
3. Business interpretation: Dans Mini Dogs has the highest number of violations, significantly surpassing the others, indicating potential issues with compliance or food safety practices.
4. Limitation: The SQL result only provides data for 2025 and does not include historical trends or comparisons with previous years.


The text-to-SQL generated result differed from the predefined SQL function result. Both approaches identified Dan’s Mini Dogs as the business with the highest number of violation records, but the text-to-SQL result reported 5 more violation records than the predefined SQL function. In addition, the next two businesses returned by the text-to-SQL query did not match the predefined SQL output.

The main reason for this difference was that the LLM-generated SQL included records with result_group = 'pass', even though the prompt instructed the model to exclude pass records. This shows that prompt instructions alone do not always guarantee that important business rules will be applied correctly.

This benchmark demonstrates that LLM-generated text-to-SQL can be flexible, but predefined SQL functions are more reliable for repeatable analytics because the filtering logic and business rules are encoded directly in the function.

text to sql LLM test 2:

In [271]:
text_sql_result2 = run_text_to_sql_benchmark(user_question_2)

print(text_sql_result2["generated_sql"])
print(text_sql_result2["summary"])


SELECT violdesc, COUNT(*) AS violation_count
FROM inspections
WHERE year = 2024 AND violdesc IS NOT NULL AND violdesc <> ''
GROUP BY violdesc
ORDER BY violation_count DESC;
1. Direct answer: The most common violation types in 2024 are related to nonfood contact surfaces and pest control.
2. Key evidence from SQL result: 
   - Nonfood Contact Surfaces (C): 2124 violations
   - Controlling Pests (Pf): 1506 violations
3. Business interpretation: There is a significant focus on maintaining cleanliness of nonfood contact surfaces and effective pest control, indicating potential areas for improvement in food safety practices.
4. Limitation: The data is limited to the year 2024 and does not provide historical context or trends over time.


text to sql LLM test 3:

In [332]:
text_sql_result3 = run_text_to_sql_benchmark(user_question_3)

print(text_sql_result3["generated_sql"])
print(text_sql_result3["summary"])


SELECT year, severity_level, COUNT(*) AS violation_volume
FROM inspections
WHERE year >= 2016 AND violdesc IS NOT NULL
GROUP BY year, severity_level
ORDER BY year, severity_level;
1. Direct answer: The violation volume by severity levels shows fluctuations, with high severity violations generally decreasing over time, while low severity violations remain significant but vary year to year.

2. Key evidence from SQL result:
   - High severity violations peaked in 2018 (10,235) and have generally declined to 1,836 in 2026.
   - Low severity violations have fluctuated, with a high of 39,892 in 2016 and a recent count of 24,225 in 2025.

3. Business interpretation:
   - Focus on reducing high severity violations should remain a priority, as their volume has decreased but still poses a risk.
   - Continuous monitoring of low severity violations is essential due to their consistently high volume, indicating potential compliance issues.

4. Limitation: The SQL result does not provide a compreh

In [339]:
text_sql_result4 = run_text_to_sql_benchmark(user_question_4)

print(text_sql_result4["generated_sql"])
print(text_sql_result4["summary"])

SELECT businessname, 
       address, 
       zip, 
       ROUND(AVG(severity_score) * 0.6 + AVG(risk_score) * 0.3 + LOG(COUNT(*) + 1) * 0.1, 2) AS overall_violation_score
FROM inspections
WHERE year = 2025 
  AND violdesc IS NOT NULL 
  AND violdesc <> ''
  AND result_group <> 'pass'
GROUP BY businessname, address, zip
ORDER BY overall_violation_score DESC;
1. Direct answer: The businesses with the highest overall serious violation risk in 2025 requiring follow-up are Lucy Ethopian Cafe, Lordya Gourmet, and Irashai Sushi and Teriyaki.

2. Key evidence from SQL result: 
   - Lucy Ethopian Cafe: overall_violation_score 2.78
   - Lordya Gourmet: overall_violation_score 2.76
   - Irashai Sushi and Teriyaki: overall_violation_score 2.75

3. Business interpretation:
   - These businesses exhibit the highest violation scores, indicating a significant risk that may affect food safety compliance.
   - Follow-up actions should prioritize inspections and corrective measures to mitigate potential

The text-to-SQL generated result differed from the predefined SQL function result. Thougth it detect the serverty level catetegoy, however, It doesn't have capability to detect aggregation needed and gerenate high, low, medium count seperately, insteadl it used the toltal counts. Therefore it conclude a wrong resutls saying 

In [252]:
def run_chart_generation_benchmark(question, result_df):
    generated_chart_code = chart_code_chain.invoke({
        "question": question,
        "columns": list(result_df.columns)
    }).strip()

    return {
        "question": question,
        "generated_chart_code": generated_chart_code
    }

In [ ]:
sql_result = run_text_to_sql_benchmark(question_sql1)

chart_benchmark = run_chart_generation_benchmark(
    sql_result["generated_sql"],
    sql_result["data"]
)

print(sql_result["generated_sql"])
print(chart_benchmark["generated_chart_code"])

SELECT year, severity_level, COUNT(*) AS violation_count
FROM inspections
WHERE year >= 2016 AND violdesc IS NOT NULL
GROUP BY year, severity_level
ORDER BY year, severity_level;
import plotly.express as px

fig = px.line(df, x='year', y='violation_count', color='severity_level', 
              title='Violation Record Count Trends by Year and Severity (2016 - Present)',
              labels={'violation_count': 'Violation Count', 'year': 'Year'})
fig.show()


The LLM-generated chart code failed because it referenced a column name that was not present in the dataframe. 

Plain LLM vs RAG

In [301]:
# Plain LLM baseline: no SQL, no RAG, no retrieved records

plain_llm_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a food safety inspection analyst.
Answer the user question based on boston area food inspection records.
Keep the answer concise.
"""),
    ("human", """
User question:
{question}
""")
])

plain_llm_chain = plain_llm_prompt | llm | StrOutputParser()

In [ ]:
def run_pure_llm(question, sample_size=20):
    raw_sample = rag_df.sample(
        sample_size,
        random_state=42
    )["rag_text"].to_string(index=False)

    response = plain_llm_chain.invoke({
        "question": question,
        "raw_sample": raw_sample
    })

    return {
        "question": question,
        "raw_sample": raw_sample,
        "response": response
    }

In [375]:

pure_llm_result_2 = plain_llm_chain.invoke({
        "question": user_question_9,
    })

pure_llm_result_2
print(pure_llm_result_2)

Recurring corrective actions for low-temperature dish machines in Boston inspection comments often include:

1. **Insufficient Sanitizer Concentration**: Ensure that the sanitizer is at the correct concentration level as per manufacturer guidelines. Regularly check and adjust the chemical levels.

2. **Improper Temperature Settings**: Verify that the machine is operating at the required temperatures for effective sanitization. Adjust settings or repair the machine as necessary.

3. **Maintenance Issues**: Schedule regular maintenance and servicing of the dish machine to prevent breakdowns and ensure optimal performance.

4. **Staff Training**: Provide training for staff on proper operation and monitoring of the dish machine to ensure compliance with health standards.

Implementing these corrective actions can help maintain compliance and ensure food safety.


In [377]:
pure_llm_result_3 = plain_llm_chain.invoke({
        "question": user_question_8,
    })

pure_llm_result_3
print(pure_llm_result_3)

In the Boston area, common violations related to sanitizer or dish machine issues include inadequate sanitizer concentration, improper temperature settings, and lack of maintenance. 

Practical corrective actions include:

1. **Sanitizer Concentration**: Regularly test sanitizer levels using test strips and adjust concentrations according to manufacturer guidelines.
   
2. **Dish Machine Temperature**: Ensure that the dish machine reaches the required temperature for sanitizing (typically 180°F for high-temp machines). Regularly check and calibrate thermometers.

3. **Maintenance**: Schedule routine maintenance for dish machines to prevent malfunctions. Keep a log of maintenance activities and repairs.

4. **Training Staff**: Train staff on proper use and monitoring of dish machines and sanitizers to ensure compliance with health standards.

5. **Documentation**: Maintain records of sanitizer tests and machine maintenance to demonstrate compliance during inspections. 

Implementing the

Observation comparison:

1. Based on the comparison above performed on question, Plain LLM: gives generic best-practice actions.
RAG + LLM: gives actions grounded in actual Boston records with sepecific anwsers eg, sepecific desired temperature. It captures specific records
: specific machine conditions, sanitizer not registering, machine taken out of service

2. The LLM generated failed to generate chart, because it used an incorrect column name. This shows why predefined chart functions are more reliable for reproducible analytics.

3. The predefined SQL function and the LLM-generated text-to-SQL query produced different analytical results for the same business question, example: regarding yearly high-severity violation trends:

    The predefined SQL function reported that high-severity violations peaked in 2016 with 6,758 records and declined to 1,820 records by 2020. In contrast, the LLM-generated text-to-SQL approach reported a higher peak in 2018 with 10,235 records.

    Even we defined the business rules do not include pass in violation record, the generated sql still includes those record. Prompt instructions alone do not guarantee enforcement of business logic. Possible reason includes:
    - prompts are long
    - multiple rules exist
    - the question itself does not explicitly mention the rule
    - the model optimizes for 'convincing sql'

In [315]:
benchmark_df = pd.DataFrame({
    "Stack": [
        "Text-to-SQL",
        "Predefined SQL function"
    ],
    "reliability": [
        "Flexible but may generate inconsistent or incorrect SQL",
        "Deterministic and repeatable"
    ],
    "uses_business_rules": [
        "Define schema and business rules clearly",
        "Not all business rulse are used as prompt"
    ],
    "result_rows": [
        len(text_to_sql_df),
        len(rag_df)
    ],
    "summary": [
        text_to_sql_summary,
        summary
    ]
})

benchmark_df

,Stack,reliability,uses_business_rules,result_rows,summary
0,Text-to-SQL,Flexible but may generate inconsistent or incorrect SQL,Define schema and business rules clearly,41,"1. Direct answer: The SQL result does not provide specific businesses with violations for 2025.\n2. Key evidence from SQL result: The data shows violation counts for different severity levels in 2025, but lacks business identification.\n3. Business interpretation: Without business-specific data, it's impossible to identify which businesses are most problematic in terms of violations.\n4. Limitation: The SQL result is limited as it does not include business names or identifiers related to the violations."
1,Predefined SQL function,Deterministic and repeatable,Not all business rulse are used as prompt,819596,"1. Direct answer: Violation record counts by year and severity show a decline in high and low severity violations from 2016 to 2019, with medium severity remaining relatively stable.\n\n2. Key evidence from SQL result:\n - High severity: 6758 (2016) to 4022 (2019)\n - Low severity: 28597 (2016) to 21258 (2017) to 26363 (2018)\n - Medium severity: 2181 (2016) to 2290 (2018)\n\n3. Business interpretation: There is a notable decrease in high severity violations over the years, indicating potential improvements in food safety practices. However, low severity violations show fluctuations, suggesting ongoing challenges.\n\n4. Limitation: The data only covers up to 2019, limiting insights into trends beyond that year."


data source link:

https://data.boston.gov/dataset/food-establishment-inspections

https://supermemory.ai/blog/best-open-source-embedding-models-benchmarked-and-ranked/